# Exercices — Reshaping

Trois exercices progressifs de mise en forme de données (pivot, melt, crosstab)
exclusivement sur le dataset des accidents de la route.

> **Consigne** : écrivez votre réponse dans la cellule vide fournie.
> La solution est cachée dans le bloc `<details>` — consultez-la après avoir essayé.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from _data import load_accidents, load_accidents_usagers

df = load_accidents()          # caractéristiques + lieux (left join)
usagers = load_accidents_usagers()

print("df      :", df.shape)
print("usagers :", usagers.shape)

## Exercice 1 — Tableau croisé : accidents par mois et luminosité

Construisez un tableau croisé qui montre le **nombre d'accidents**
avec en **lignes** les mois (`mois`, 1–12)
et en **colonnes** les conditions de luminosité (`lum`) :

| lum | Signification |
|-----|---------------|
| 1 | Plein jour |
| 2 | Crépuscule ou aube |
| 3 | Nuit sans éclairage |
| 4 | Nuit avec éclairage non allumé |
| 5 | Nuit avec éclairage allumé |

**Étape 1** : utilisez `pd.pivot_table()` pour construire ce tableau.

**Étape 2** : normalisez par ligne (pourcentage par mois) pour comparer la répartition indépendamment du volume total.

> **Indice** : pour compter les lignes, utilisez `aggfunc="count"` et une colonne de comptage comme `Num_Acc`.

In [ ]:
# Étape 1 — tableau de comptage brut


In [ ]:
# Étape 2 — normalisation par ligne (% par mois)


<details><summary>Solution</summary>

```python
# Étape 1
pivot = pd.pivot_table(
    df,
    values="Num_Acc",
    index="mois",
    columns="lum",
    aggfunc="count",
    fill_value=0,
)
pivot
```

```python
# Étape 2 — diviser chaque ligne par le total de la ligne
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).round(3) * 100
pivot_pct
```

**Alternative avec `pd.crosstab()` :**
```python
pd.crosstab(df["mois"], df["lum"], normalize="index").round(3) * 100
```

**Points à retenir :**
- `fill_value=0` remplace les NaN par 0 (ici : aucun accident dans cette combinaison).
- `.div(pivot.sum(axis=1), axis=0)` : on divise chaque ligne par sa somme. `axis=0` aligne l'opération sur les lignes.
- `pd.crosstab` avec `normalize="index"` fait la même chose en une seule ligne.

</details>

## Exercice 2 — Melt : dérouler les équipements de sécurité

Dans `usagers`, les colonnes `secu1`, `secu2`, `secu3` représentent les équipements de sécurité
portés par chaque usager (ceinture, casque, gilet…). Un usager peut en avoir jusqu'à 3.

Ce format *wide* rend difficile de compter le nombre total de fois où chaque équipement apparaît.

**Objectif** :
1. Avec `.melt()`, transformez les 3 colonnes `secu1`/`secu2`/`secu3` en une seule colonne `secu`
   (format *long*).
2. Supprimez les lignes où `secu` est NaN ou vaut 0 (non renseigné).
3. Comptez le nombre d'occurrences de chaque valeur de `secu`.

> **Rappel** : `.melt(id_vars=[...], value_vars=[...], var_name="...", value_name="...")`

In [ ]:
# Votre réponse ici


<details><summary>Solution</summary>

```python
(
    usagers
    .melt(
        id_vars=["Num_Acc", "id_usager"],
        value_vars=["secu1", "secu2", "secu3"],
        var_name="rang_secu",
        value_name="secu",
    )
    .dropna(subset=["secu"])
    .query("secu != 0")
    .groupby("secu")
    .agg(n=("Num_Acc", "count"))
    .sort_values("n", ascending=False)
)
```

**Points à retenir :**
- `id_vars` : colonnes qui restent fixes — elles sont répétées pour chaque ligne produite.
- `value_vars` : les colonnes à "empiler" en une seule.
- Après melt, le DataFrame a 3× plus de lignes (une par colonne secu).
- `query("secu != 0")` filtre les codes 0 qui signifient "non renseigné".

</details>

## Exercice 3 — Stack / Unstack : gravité par type d'usager et département

Cet exercice combine groupby, pivot et stack pour produire un tableau d'analyse multi-niveaux.

Le champ `catu` dans `usagers` indique le type d'usager :
- 1 = conducteur
- 2 = passager
- 3 = piéton

**Objectif** :
Construire un tableau qui montre, pour les **10 départements les plus accidentogènes**,
la **proportion de tués** (`grav == 2`) parmi les usagers,
**par type d'usager** (`catu`).

Format attendu :
```
catu           1         2         3
dep
13        0.012     0.008     0.015
31        ...
...
```

Étapes suggérées :
1. Ajouter une colonne `tue` (1 si `grav == 2`, 0 sinon) dans `usagers`.
2. Merger avec `df` pour récupérer `dep`.
3. Filtrer sur les 10 départements les plus accidentogènes.
4. Utiliser `pivot_table` ou `groupby` + `unstack` pour obtenir le format souhaité.

> **Niveau avancé** : pensez à filtrer les départements *avant* de construire le pivot
> — c'est beaucoup plus efficace que de filtrer après.

In [ ]:
# Votre réponse ici


<details><summary>Solution</summary>

```python
# Top 10 départements (sur les accidents)
top10_dep = (
    df.groupby("dep")["Num_Acc"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .index
)

# Construire le tableau
(
    usagers
    .assign(tue=lambda df_: (df_["grav"] == 2).astype(int))
    .merge(df[["Num_Acc", "dep"]], on="Num_Acc", how="left")
    .query("dep in @top10_dep")
    .groupby(["dep", "catu"])
    .agg(taux_tue=("tue", "mean"))
    .round(4)
    .unstack("catu")
    .droplevel(0, axis=1)  # enlever le niveau "taux_tue" dans les colonnes
)
```

**Variante avec `pivot_table` :**
```python
base = (
    usagers
    .assign(tue=lambda df_: (df_["grav"] == 2).astype(int))
    .merge(df[["Num_Acc", "dep"]], on="Num_Acc", how="left")
    .query("dep in @top10_dep")
)

pd.pivot_table(
    base,
    values="tue",
    index="dep",
    columns="catu",
    aggfunc="mean",
).round(4)
```

**Points à retenir :**
- `(df_["grav"] == 2).astype(int)` crée une colonne binaire 0/1 — la moyenne de cette colonne est directement le taux.
- `.query("dep in @top10_dep")` : `@` dans `.query()` référence une variable Python locale.
- `.unstack("catu")` pivote le niveau `catu` de l'index vers les colonnes.
- `.droplevel(0, axis=1)` supprime le niveau redondant `"taux_tue"` dans le MultiIndex de colonnes.
- `pivot_table` fait la même chose mais en un seul appel.

</details>